# QECCT Student: 온디바이스 SoC를 위한 Knowledge Distillation 경량화
**목적**: Teacher QECCT (논문 재현)로부터 경량화된 Student 모델을 학습하고, Pruning + Quantization을 추가 적용  
**출력**: 논문 삽입용 비교 플롯 (LER, 파라미터, 압축률, 지연시간)

In [ ]:
# 필요 시: !pip install pymatching
import sys, os, time, json
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

from qecct_models import ToricCode, QECCT, QECCTLoss, compute_ber, compute_ler
from qecct_train import Trainer, evaluate_mwpm
from qecct_student import (QECCTStudent, KDLoss, KDTrainer,
                            apply_structured_pruning, apply_dynamic_quantization,
                            measure_model_size,
                            plot_model_comparison_bar, plot_ler_teacher_student,
                            plot_kd_training_curves, plot_compression_summary_table,
                            plot_latency_comparison)

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Device: {device} | PyTorch: {torch.__version__} | {datetime.now()}")

## 1. 실험 설정

In [ ]:
CONFIG = {
    # Teacher (논문 기본)
    'teacher_N': 6, 'teacher_d': 128, 'teacher_heads': 8,
    # Student (경량)
    'student_N': 2, 'student_d': 32, 'student_heads': 4,
    'use_linear_attn': False,  # True면 Linear Attention 사용
    # KD 설정
    'temperature': 3.0,
    'alpha_task': 0.5, 'alpha_kd': 0.3, 'alpha_attn': 0.2,
    # 학습
    'lr': 5e-4, 'lr_min': 5e-7, 'batch_size': 512,
    'n_epochs': 200, 'n_batches': 5000,
    'lambda_ber': 0.5, 'lambda_ler': 1.0, 'lambda_g': 0.5,
    # 실험
    'L': 3,  # 격자 크기
    'noise_type': 'independent',
    'p_train_range': (0.01, 0.15),
    'p_eval': [0.02, 0.04, 0.06, 0.08, 0.10, 0.12, 0.14],
    'eval_every': 20, 'eval_n_samples': 10000,
    # Pruning
    'prune_ratio': 0.5,
    # Quick-run
    'quick_run': True,
}

if CONFIG['quick_run']:
    CONFIG.update({
        'teacher_N': 3, 'teacher_d': 64, 'teacher_heads': 4,
        'student_N': 2, 'student_d': 32, 'student_heads': 4,
        'n_epochs': 5, 'n_batches': 50,
        'eval_every': 5, 'eval_n_samples': 500,
    })
    print("⚡ Quick-run mode")

for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## 2. Teacher 모델 준비

In [ ]:
code_obj = ToricCode(CONFIG['L'])
print(f"Toric Code L={CONFIG['L']}: n={code_obj.n}, n_s={code_obj.n_s}")

# Teacher 모델 (사전 학습된 가중치 로드 또는 새로 학습)
teacher = QECCT(code_obj, N=CONFIG['teacher_N'], d_model=CONFIG['teacher_d'],
                n_heads=CONFIG['teacher_heads']).to(device)

teacher_params = sum(p.numel() for p in teacher.parameters())
print(f"Teacher params: {teacher_params:,}")

# Teacher 학습 (사전 학습 가중치가 없을 경우)
teacher_ckpt = f"qecct_L{CONFIG['L']}.pt"
if os.path.exists(teacher_ckpt):
    teacher.load_state_dict(torch.load(teacher_ckpt, map_location=device))
    print(f"✅ Teacher loaded from {teacher_ckpt}")
else:
    print("Teacher 학습 시작...")
    t_trainer = Trainer(teacher, code_obj, device,
                        lr=CONFIG['lr'], batch_size=CONFIG['batch_size'],
                        noise_type=CONFIG['noise_type'], p_range=CONFIG['p_train_range'])
    t_history = t_trainer.train(n_epochs=CONFIG['n_epochs'],
                                n_batches_per_epoch=CONFIG['n_batches'],
                                eval_every=CONFIG['eval_every'],
                                eval_p_range=CONFIG['p_eval'],
                                eval_n_samples=CONFIG['eval_n_samples'])
    torch.save(teacher.state_dict(), teacher_ckpt)
    print(f"✅ Teacher saved: {teacher_ckpt}")

# Teacher 평가
teacher.eval()
t_trainer_eval = Trainer(teacher, code_obj, device, noise_type=CONFIG['noise_type'])
teacher_results = t_trainer_eval.evaluate(CONFIG['p_eval'], CONFIG['eval_n_samples'])
print("\nTeacher LER:", {f"p={p:.2f}": f"{ler:.4f}" for p, ler in zip(CONFIG['p_eval'], teacher_results['ler'])})

## 3. Student 모델 학습 (Knowledge Distillation)

In [ ]:
# Student 모델 생성
student = QECCTStudent(code_obj, N=CONFIG['student_N'], d_model=CONFIG['student_d'],
                       n_heads=CONFIG['student_heads'],
                       use_linear_attn=CONFIG['use_linear_attn']).to(device)

student_params = sum(p.numel() for p in student.parameters())
print(f"Student params: {student_params:,} ({student_params/teacher_params:.1%} of Teacher)")

# KD 학습
kd_trainer = KDTrainer(
    teacher=teacher, student=student, code=code_obj, device=device,
    lr=CONFIG['lr'], lr_min=CONFIG['lr_min'], batch_size=CONFIG['batch_size'],
    noise_type=CONFIG['noise_type'], p_range=CONFIG['p_train_range'],
    temperature=CONFIG['temperature'],
    alpha_task=CONFIG['alpha_task'], alpha_kd=CONFIG['alpha_kd'], alpha_attn=CONFIG['alpha_attn']
)

kd_history = kd_trainer.train(
    n_epochs=CONFIG['n_epochs'], n_batches_per_epoch=CONFIG['n_batches'],
    eval_every=CONFIG['eval_every'], eval_p_range=CONFIG['p_eval'],
    eval_n_samples=CONFIG['eval_n_samples']
)

# Student 평가
student_results = kd_trainer.evaluate(CONFIG['p_eval'], CONFIG['eval_n_samples'])
print("\nStudent LER:", {f"p={p:.2f}": f"{ler:.4f}" for p, ler in zip(CONFIG['p_eval'], student_results['ler'])})
torch.save(student.state_dict(), f"qecct_student_L{CONFIG['L']}.pt")
print("✅ Student saved")

## 4. 구조적 Pruning 적용

In [ ]:
# Pruning
pruned_student, prune_stats = apply_structured_pruning(student, CONFIG['prune_ratio'])
pruned_student = pruned_student.to(device)

print(f"Pruning Stats:")
print(f"  Before: {prune_stats['total_before']:,} params")
print(f"  After (nonzero): {prune_stats['nonzero_after']:,} params")
print(f"  Sparsity: {prune_stats['sparsity']:.1%}")
print(f"  Layers pruned: {prune_stats['layers_pruned']}")

# Pruned model 평가
pruned_student.eval()
# Quick evaluator using student's eval method
from qecct_student import KDTrainer
pruned_trainer = KDTrainer(teacher, pruned_student, code_obj, device,
                           noise_type=CONFIG['noise_type'])
pruned_results = pruned_trainer.evaluate(CONFIG['p_eval'], CONFIG['eval_n_samples'])
print("\nPruned LER:", {f"p={p:.2f}": f"{ler:.4f}" for p, ler in zip(CONFIG['p_eval'], pruned_results['ler'])})

## 5. INT8 양자화

In [ ]:
import copy

# 양자화 (CPU에서만 지원)
student_cpu = copy.deepcopy(student).cpu()

# 모델 크기 비교
size_teacher = measure_model_size(teacher.cpu()) / 1024
size_student = measure_model_size(student_cpu) / 1024
teacher.to(device)

try:
    quantized_student = apply_dynamic_quantization(student_cpu)
    size_quantized = measure_model_size(quantized_student) / 1024
    print(f"Teacher size:    {size_teacher:.1f} KB")
    print(f"Student size:    {size_student:.1f} KB")
    print(f"Quantized size:  {size_quantized:.1f} KB")
    print(f"Compression:     {size_teacher/size_quantized:.1f}\u00d7")
except Exception as e:
    print(f"Quantization skipped: {e}")
    size_quantized = size_student
    quantized_student = student_cpu

# 양자화 모델 실제 평가 (정확한 LER 측정)
print("\nEvaluating quantized model...")
quantized_results = {'p': CONFIG['p_eval'], 'ber': [], 'ler': []}
for p in CONFIG['p_eval']:
    noise_q = code_obj.sample_independent_noise(p, CONFIG['eval_n_samples'])
    syns_q = np.array([code_obj.get_syndrome(noise_q[i]) for i in range(CONFIG['eval_n_samples'])])
    preds_q = []
    for i in range(0, CONFIG['eval_n_samples'], 512):
        s_b = torch.tensor(syns_q[i:i+512], dtype=torch.float32)
        with torch.no_grad():
            preds_q.append(quantized_student(s_b)['prediction'].numpy())
    preds_q = np.concatenate(preds_q, axis=0)
    quantized_results['ber'].append(compute_ber(preds_q, noise_q))
    quantized_results['ler'].append(compute_ler(preds_q, noise_q, code_obj.L_matrix))

print("\nQuantized LER:", {f"p={p:.2f}": f"{ler:.4f}" for p, ler in zip(CONFIG['p_eval'], quantized_results['ler'])})

## 6. 추론 지연시간 측정

In [ ]:
# 추론 속도 벤치마크
def benchmark_latency(model, syndrome_tensor, n_runs=100):
    model.eval()
    dev = next(model.parameters()).device
    s = syndrome_tensor.to(dev)
    # Warmup
    for _ in range(10):
        with torch.no_grad():
            _ = model(s)
    # Measure
    if dev.type == 'cuda':
        torch.cuda.synchronize()
    start = time.time()
    for _ in range(n_runs):
        with torch.no_grad():
            _ = model(s)
    if dev.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = (time.time() - start) / n_runs
    return elapsed * 1000  # ms

# 테스트 데이터
test_noise = code_obj.sample_independent_noise(0.1, 1)
test_syn = np.array([code_obj.get_syndrome(test_noise[0])])
test_syn_t = torch.tensor(test_syn, dtype=torch.float32)

latency_teacher = benchmark_latency(teacher.to(device), test_syn_t)
latency_student = benchmark_latency(student.to(device), test_syn_t)
latency_quantized = benchmark_latency(quantized_student, test_syn_t.cpu())

print(f"Teacher latency:   {latency_teacher:.3f} ms/sample")
print(f"Student latency:   {latency_student:.3f} ms/sample")
print(f"Quantized latency: {latency_quantized:.3f} ms/sample")
print(f"Speedup (Student): {latency_teacher/latency_student:.1f}×")

## 7. MWPM 베이스라인

In [ ]:
mwpm_results = evaluate_mwpm(code_obj, CONFIG['p_eval'],
                           noise_type=CONFIG['noise_type'],
                           n_samples=CONFIG['eval_n_samples'])
print("MWPM LER:", {f"p={p:.2f}": f"{ler:.4f}" for p, ler in zip(CONFIG['p_eval'], mwpm_results['ler'])})

## 8. 논문용 플롯

### 8.1 LER 성능 비교 (Figure)
Teacher (QECCT) vs Student (KD) vs Student+Pruning vs MWPM

In [ ]:
plot_ler_teacher_student(
    teacher_results, student_results,
    mwpm_results=mwpm_results,
    pruned_results=pruned_results,
    code_L=CONFIG['L'],
    save_path='fig_ler_comparison.png'
)
print("✅ Saved: fig_ler_comparison.png")

### 8.2 모델 크기 비교 (Figure)

In [ ]:
plot_model_comparison_bar(
    teacher_params=teacher_params,
    student_params=student_params,
    pruned_params=prune_stats['nonzero_after'],
    save_path='fig_model_size.png'
)
print("✅ Saved: fig_model_size.png")

### 8.3 KD 학습 곡선 (Figure)

In [ ]:
plot_kd_training_curves(kd_history, save_path='fig_kd_training.png')
print("✅ Saved: fig_kd_training.png")

### 8.4 경량화 요약 테이블 (Table)

In [ ]:
# LER @p=0.10 수집
def get_ler_at_p(results, target_p=0.10):
    for i, p in enumerate(results['p']):
        if abs(p - target_p) < 0.005:
            return results['ler'][i]
    return results['ler'][len(results['p'])//2]

metrics = {
    'MWPM': {
        'params': 'N/A', 'size_kb': 0,
        'ler_010': get_ler_at_p(mwpm_results), 'compression': 1.0
    },
    'Teacher (QECCT)': {
        'params': teacher_params, 'size_kb': size_teacher,
        'ler_010': get_ler_at_p(teacher_results), 'compression': 1.0
    },
    'Student (KD)': {
        'params': student_params, 'size_kb': size_student,
        'ler_010': get_ler_at_p(student_results),
        'compression': teacher_params / student_params
    },
    'Student+Prune': {
        'params': prune_stats['nonzero_after'], 'size_kb': size_student * (1 - prune_stats['sparsity']),
        'ler_010': get_ler_at_p(pruned_results),
        'compression': teacher_params / prune_stats['nonzero_after']
    },
    'Student+Quant': {
        'params': student_params, 'size_kb': size_quantized,
        'ler_010': get_ler_at_p(quantized_results),  # 실제 양자화 모델 평가 결과 사용
        'compression': size_teacher / size_quantized
    },
}

plot_compression_summary_table(metrics, save_path='fig_compression_table.png')
print("\u2705 Saved: fig_compression_table.png")

### 8.5 추론 속도 비교 (Figure)

In [ ]:
plot_latency_comparison(
    methods=['Teacher (QECCT)', 'Student (KD)', 'Student (INT8)'],
    latencies_ms=[latency_teacher, latency_student, latency_quantized],
    save_path='fig_latency.png'
)
print("✅ Saved: fig_latency.png")

### 8.6 논문용 종합 Figure (2-panel)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: LER comparison
ax = axes[0]
if mwpm_results:
    ax.semilogy(mwpm_results['p'], mwpm_results['ler'], 'D-',
                color='#9E9E9E', label='MWPM', linewidth=2, markersize=6)
ax.semilogy(teacher_results['p'], teacher_results['ler'], 'o-',
            color='#2196F3', label='Teacher (QECCT)', linewidth=2.5, markersize=6)
ax.semilogy(student_results['p'], student_results['ler'], 's--',
            color='#4CAF50', label='Student (KD)', linewidth=2.5, markersize=6)
ax.semilogy(pruned_results['p'], pruned_results['ler'], '^:',
            color='#FF9800', label='Student+Pruning', linewidth=2, markersize=6)
ax.set_xlabel('Physical Error Rate (p)', fontsize=12)
ax.set_ylabel('Logical Error Rate (LER)', fontsize=12)
ax.set_title(f'(a) LER Performance (Toric L={CONFIG["L"]})', fontsize=13, fontweight='bold')
ax.legend(fontsize=9.5, framealpha=0.9)
ax.grid(True, alpha=0.3, which='both')

# Right: Model size + Speedup
ax2 = axes[1]
methods = ['Teacher', 'Student\n(KD)', 'Student\n+Prune']
params_list = [teacher_params, student_params, prune_stats['nonzero_after']]
colors = ['#2196F3', '#4CAF50', '#FF9800']
bars = ax2.bar(methods, params_list, color=colors, edgecolor='white', width=0.55)
for bar, v in zip(bars, params_list):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(params_list)*0.03,
             f'{v:,}', ha='center', fontweight='bold', fontsize=9)
ax2.set_ylabel('Parameters', fontsize=12)
ax2.set_title('(b) Model Complexity', fontsize=13, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# Compression annotations
for i, (m, p) in enumerate(zip(methods, params_list)):
    ratio = teacher_params / p
    if i > 0:
        ax2.annotate(f'{ratio:.1f}× smaller', xy=(i, p),
                     xytext=(i+0.3, p + max(params_list)*0.15),
                     fontsize=9, color=colors[i], fontweight='bold',
                     arrowprops=dict(arrowstyle='->', color=colors[i], lw=1.5))

plt.tight_layout()
plt.savefig('fig_paper_combined.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: fig_paper_combined.png")

## 9. 결과 저장

In [ ]:
results_all = {
    'config': {k: str(v) for k, v in CONFIG.items()},
    'teacher': {'params': teacher_params, 'ler': teacher_results['ler'], 'ber': teacher_results['ber']},
    'student': {'params': student_params, 'ler': student_results['ler'], 'ber': student_results['ber']},
    'pruned': {'params': prune_stats['nonzero_after'], 'sparsity': prune_stats['sparsity'],
               'ler': pruned_results['ler'], 'ber': pruned_results['ber']},
    'mwpm': {'ler': mwpm_results['ler'], 'ber': mwpm_results['ber']},
    'p_values': CONFIG['p_eval'],
    'latency_ms': {'teacher': latency_teacher, 'student': latency_student, 'quantized': latency_quantized},
    'sizes_kb': {'teacher': size_teacher, 'student': size_student, 'quantized': size_quantized},
}

with open('student_results.json', 'w') as f:
    json.dump(results_all, f, indent=2, default=str)
print("✅ Saved: student_results.json")

print("\n" + "="*60)
print("📊 생성된 논문용 Figure 파일:")
print("  fig_ler_comparison.png    - LER 성능 비교")
print("  fig_model_size.png        - 모델 크기/압축률")
print("  fig_kd_training.png       - KD 학습 곡선")
print("  fig_compression_table.png - 경량화 요약 테이블")
print("  fig_latency.png           - 추론 속도 비교")
print("  fig_paper_combined.png    - 종합 2-panel Figure")
print("="*60)